In [45]:
import pandas as pd

# Load the CSV file
df = pd.read_csv('data/mfrc_only_moral.csv')

# List unique labels
unique_labels = df['annotation'].unique()
unique_labels = [x for x in unique_labels if ',' not in x]
print(', '.join(map(str, unique_labels)))

Purity, Equality, Authority, Care, Loyalty, Proportionality


In [46]:
import pandas as pd
df = pd.read_csv('data/d3-ratings.csv')



In [47]:
df.rating_raw.value_counts()

rating_raw
 0    69960
 1    28637
 2    18855
 4    14772
 3    12506
-1     5972
Name: count, dtype: int64

In [48]:
from datasets import load_dataset

ds = load_dataset('aequa-tech/CADE')

/opt/homebrew/anaconda3/envs/nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [49]:
ds = ds['train'].to_pandas()

In [50]:
ds = ds.drop_duplicates(subset=['text_id'])

In [51]:
df = ds

df.to_csv('/Users/marco/Documents/aequa/calibration-bias/data/cade/corpus.csv', index=False)

In [52]:
import json,numpy as np

with open('output/internal_validity_outputs.json') as f:
    jsn = json.load(f)

In [53]:
from scipy.special import rel_entr
from scipy.spatial.distance import jensenshannon

keys = list(jsn.keys())

val = 'care'
care = list()
for k in keys:
    l = list()
    for v in jsn[k][val]:
        #probs = list(v['probs'].keys())
        
        lab = max(v['probs'], key=v['probs'].get)
        l.append(lab)
    
    '''l = np.sum(np.array(l), axis=0)
        
    l = [x / np.sum(l) for x in l]'''
    
        
        

        

    care.append(l)



        

In [54]:
care

[['3', '3', '3', '3', '3', '3'],
 ['4', '4', '4', '4', '3', '2'],
 ['5', '5', '5', '5', '4', '5'],
 ['5', '5', '5', '3', '4', '4'],
 ['5', '5', '5', '5', '5', '5'],
 ['1', '1', '1', '1', '1', '1']]

In [55]:
from itertools import combinations

care = list(combinations(care, 2))

for item in care:

    js = jensenshannon(item[0], item[1])
    print(js)

UFuncTypeError: ufunc 'add' did not contain a loop with signature matching types (dtype('<U1'), dtype('<U1')) -> None

In [ ]:
care

[array([1.88927036e-03, 1.00102694e-01, 8.02613464e-01, 9.51902093e-02,
        2.04362069e-04]),
 array([5.88502136e-03, 5.06893448e-01, 1.56296451e-01, 3.30756082e-01,
        1.68997610e-04]),
 array([0.00000000e+00, 1.55166051e-05, 3.94573519e-01, 5.41149411e-01,
        6.42615531e-02]),
 array([4.86737128e-07, 1.09075804e-03, 3.60467971e-01, 5.91165646e-01,
        4.72751383e-02]),
 array([8.57165036e-06, 1.78226390e-05, 2.23710084e-02, 4.64198065e-01,
        5.13404533e-01]),
 array([0.5686396 , 0.17958962, 0.09557559, 0.14796773, 0.00822747])]

In [ ]:
with open('output/external_offensiveness_outputs.json') as f:
    myjsn = json.load(f)

In [ ]:
from collections import Counter
diff = list()
for item in keys:
    l = list()
    for el in myjsn[item]:
        l.append(el['model_choice'])
        total = len(l)
    l = Counter(l)
    l = [v / total for v in l.values()]
    diff.append(np.array(l))
    



In [ ]:
diff = list(combinations(diff, 2))

for item in diff:
    try:
        js = jensenshannon(item[0], item[1])
        print(js)
    except: continue

0.1361531049235526
0.18958908300358007
0.22901536208400627
0.21733727331016506
0.05991889085555655
0.028808731957817316
0.22551965380739447
0.13083337116767274
0.3007999356257612
0.23274595192816339


In [ ]:

with open('output/free_text/Falcon3-7B-Instruct_internal_validity_external_offensiveness.json') as f:
    jsn = json.load(f)

keys = list(jsn.keys())

In [ ]:
jsn

{'model_name': 'Falcon3-7B-Instruct',
 'internal_validity': {'care': [{'test_statement': 'Caring for people who have suffered is an important virtue',
    'probs': {'1': 8.100271224975586e-05,
     '2': 2.7954578399658203e-05,
     '3': 0.00025653839111328125,
     '4': 0.08746337890625,
     '5': 0.91162109375},
    'explanation': ["4 = Agree\n\nCaring for people who have suffered is an important virtue because it reflects empathy, compassion, and a genuine desire to alleviate others' pain and suffering. It fosters a sense of community and support, which is crucial for healing and recovery. By caring for those in need, we contribute to the well-being of society as a whole and uphold values of kindness and human dignity.",
     '5 = Strongly agree\n\nCaring for people who have suffered is an important virtue because it reflects empathy, compassion, and humanity. It helps in building stronger communities, fostering a sense of belonging, and promoting healing and recovery. By showing car

### CREATE A DF WITH QUESTIONNAIRE-BASED SCORES

In [77]:
import json, pandas as pd,glob,numpy as np

scores = list()
moral_values = ['care', 'equality', 'proportionality', 'loyalty', 'authority', 'purity']

for doc in glob.glob('output/free_text/*.json'):
    with open(doc) as f:
        jsn = json.load(f)

    model = doc.split('/')[-1].split('_')[0]

    
    for value in moral_values:
        for stats in jsn['internal_validity'][value]:
            stat = stats['test_statement']
            probs = [x for x in stats['probs'].values()]
            prob = np.argmax(probs)+1
            
            scores.append({'model': model, 'moral_value': value, 'statement': stat, 'qst_score': prob})

In [83]:
import json, pandas as pd,glob

all_items =pd.read_csv('data/d3-items.csv')
moral_values = ['care', 'equality', 'proportionality', 'loyalty', 'authority', 'purity']

for doc in glob.glob('output/free_text/*.json'):
    l = list()
    with open(doc) as f:
        jsn = json.load(f)

    model = doc.split('/')[-1].split('_')[0]
    
    for stats in jsn['external_offensiveness']:
        stat = stats['test_statement']
        probs = [x for x in stats['probs'].values()]
        prob = np.argmax(probs)+1
        
        l.append({'model': model, 'text': stat, 'model_score': prob})
    pd.DataFrame(l).merge(all_items,on='text').drop(columns=['text']).to_csv(f'output/model_ratings/{model}_ratings.csv', index=False)

In [78]:
scores = pd.DataFrame(scores)

scores.to_csv('questionnaire_scores.csv',index=False)

In [ ]:
import json, pandas as pd,glob

l = list()
moral_values = ['care', 'equality', 'proportionality', 'loyalty', 'authority', 'purity']

for doc in glob.glob('output/free_text/*.json'):
    with open(doc) as f:
        jsn = json.load(f)

    model = doc.split('/')[-1].split('_')[0]

    
    for value in moral_values:
        for stats in jsn['internal_validity'][value]:
            stat = stats['test_statement']
            exp = stats['explanation'][0]
            
            l.append({'model': model, 'moral_value': value, 'statement': stat, 'explanation': exp})

In [ ]:
df = pd.DataFrame(l)
df

,model,moral_value,statement,explanation
0,Falcon3-7B-Instruct,care,Caring for people who have suffered is an impo...,5 = Strongly agree\n\nCaring for people who ha...
1,Falcon3-7B-Instruct,care,I believe that compassion for those who are su...,4 = Agree\n\nI agree with the statement becaus...
2,Falcon3-7B-Instruct,care,We should all care for people who are in emoti...,4 = Agree\n\nI agree with the statement becaus...
3,Falcon3-7B-Instruct,care,I am empathetic toward those people who have s...,4 = Agree\n\nI agree with the statement becaus...
4,Falcon3-7B-Instruct,care,Everyone should try to comfort people who are ...,4 = Agree\n\nI agree with the statement becaus...
...,...,...,...,...
139,Qwen3-8B,purity,I believe chastity is an important virtue.,"4\n\nI agree with the statement ""I believe cha..."
140,Qwen3-8B,purity,It upsets me when people use foul language lik...,4\n\nI agree with the statement because using ...
141,Qwen3-8B,purity,If I found out that an acquaintance had an unu...,3\n\nI neither agree nor disagree because the ...
142,Qwen3-8B,purity,People should try to use natural medicines rat...,3\n\nI neither agree nor disagree with the sta...


In [ ]:
import regex as re
df['redacted_explanation'] = df.explanation.apply(lambda x:x.split('**Explanation:** ')[1] if '**Explanation:** ' in x else x)
df['redacted_explanation'] = df.redacted_explanation.apply(lambda x:x.split('\nExplanation: ')[1] if '\nExplanation: ' in x else x)

df.redacted_explanation = df.redacted_explanation.apply(lambda x:re.sub(r'[0-5]','[NUMBER]', x))

df.redacted_explanation = df.redacted_explanation.apply(lambda x:re.sub(r'Agree|Disagree|agree|disagree','[EVALUATION]', x))

In [ ]:
df.to_csv('output/explanations.csv', index=False)

### clean

In [ ]:
import regex as re
df.redacted_explanation = df.redacted_explanation.apply(lambda x: re.sub(r"\[NUMBER\] \= \[EVALUATION\]","",x,re.IGNORECASE))
df.redacted_explanation = df.redacted_explanation.apply(lambda x: re.sub(r"I strongly \[EVALUATION\] with the statement|I strongly \[EVALUATION\] with the statement","",x,re.IGNORECASE))
df.redacted_explanation = df.redacted_explanation.apply(lambda x: re.sub(r"I \[EVALUATION\] with the statement because|I \[EVALUATION\] with this statement because","",x,re.IGNORECASE))
df.redacted_explanation = df.redacted_explanation.apply(lambda x: re.sub(r"I \[EVALUATION\] with the statement, because|I \[EVALUATION\] with the statement","",x,re.IGNORECASE))
df.redacted_explanation = df.redacted_explanation.apply(lambda x: re.sub(r"Neither \[EVALUATION\] nor \[EVALUATION\]","",x,re.IGNORECASE))
df.redacted_explanation = df.redacted_explanation.apply(lambda x: re.sub(r"I'm neutral on this statement because|I don't fully \[EVALUATION\]","",x, re.IGNORECASE))
df.redacted_explanation = df.redacted_explanation.apply(lambda x: re.sub(r"I neither strongly \[EVALUATION\] nor \[EVALUATION\] with the statement","",x, re.IGNORECASE))
df.redacted_explanation = df.redacted_explanation.apply(lambda x: re.sub(r"I choose to \[EVALUATION\] with this statement because","",x, re.IGNORECASE))
df.redacted_explanation = df.redacted_explanation.apply(lambda x: re.sub(r"I neither \[EVALUATION\] nor \[EVALUATION\]|I don't fully \[EVALUATION\] or \[EVALUATION\]","",x, re.IGNORECASE))
df.redacted_explanation = df.redacted_explanation.apply(lambda x: re.sub(r"\[NUMBER\] = Strongly \[EVALUATION\]","",x, re.IGNORECASE))

In [ ]:
df.to_csv('output/explanations_redacted.csv', index=False)

In [ ]:
df.iloc[0]

model                                                 Falcon3-7B-Instruct
moral_value                                                          care
statement               Caring for people who have suffered is an impo...
explanation             4 = Agree\n\nCaring for people who have suffer...
redacted_explanation    \n\nCaring for people who have suffered is an ...
Name: 0, dtype: object

In [ ]:
df = df.reset_index()

In [ ]:
df.to_csv('annotation_data.csv', index=False)

In [ ]:
import pandas as pd
df = pd.read_csv('annotation_data.csv')

In [ ]:
df = df[df.model=='Llama-3.1-8B-Instruct'].groupby(['model', 'moral_value','statement'])['redacted_explanation'].apply(list).reset_index()

In [ ]:
from random import seed,sample
df
seed(42)
df['redacted_explanation'] = df.redacted_explanation.apply(lambda x: sample(x,k=1))

In [ ]:
#df = df.drop(columns=['redacted_explanation'])
df = df.explode('redacted_explanation').reset_index(drop=True)

In [ ]:
df

,model,moral_value,statement,redacted_explanation
0,Llama-3.1-8B-Instruct,authority,I believe that one of the most important value...,[NUMBER]\n\nI [EVALUATION] with this statement...
1,Llama-3.1-8B-Instruct,authority,I feel that most traditions serve a valuable f...,[NUMBER]\n\nI [EVALUATION] with this statement...
2,Llama-3.1-8B-Instruct,authority,I think having a strong leader is good for soc...,[NUMBER]\n\nI [EVALUATION] with this statement...
3,Llama-3.1-8B-Instruct,authority,I think it is important for societies to cheri...,[NUMBER]\n\nI [EVALUATION] with this statement...
4,Llama-3.1-8B-Instruct,authority,I think obedience to parents is an important v...,[NUMBER]\n\nI [EVALUATION] with this statement...
5,Llama-3.1-8B-Instruct,authority,We all need to learn from our elders.,[NUMBER]\n\nI [EVALUATION] with this statement...
6,Llama-3.1-8B-Instruct,care,Caring for people who have suffered is an impo...,[NUMBER]\n\nI choose [NUMBER] because caring f...
7,Llama-3.1-8B-Instruct,care,Everyone should try to comfort people who are ...,[NUMBER]\n\nI [EVALUATION] with this statement...
8,Llama-3.1-8B-Instruct,care,I am empathetic toward those people who have s...,[NUMBER]\n\nI choose [NUMBER] because the stat...
9,Llama-3.1-8B-Instruct,care,I believe that compassion for those who are su...,[NUMBER]\n\nI [EVALUATION] with this statement...


In [ ]:
old = pd.read_csv('annotation_data.csv')

df = df.merge(old[['index','redacted_explanation']],left_on='sampled_explanations', right_on='redacted_explanation', how='left')

,index,model,moral_value,statement,explanation,redacted_explanation
0,0,Qwen3-8B,care,Caring for people who have suffered is an impo...,5 = Strongly agree \n**Explanation:** Caring ...,Caring for people who have suffered is a funda...
1,1,Qwen3-8B,care,Caring for people who have suffered is an impo...,5 = Strongly agree \nExplanation: Caring for ...,Caring for people who have suffered is a funda...
2,2,Qwen3-8B,care,Caring for people who have suffered is an impo...,5 = Strongly agree \n**Explanation:** Caring ...,Caring for people who have suffered is a funda...
3,3,Qwen3-8B,care,Caring for people who have suffered is an impo...,5 = Strongly agree \n**Explanation:** Caring ...,Caring for people who have suffered is a funda...
4,4,Qwen3-8B,care,Caring for people who have suffered is an impo...,5 = Strongly agree \nExplanation: Caring for ...,Caring for people who have suffered is a funda...
...,...,...,...,...,...,...
715,715,Olmo-3-1025-7B,purity,I admire people who keep their virginity until...,"4\n\nOkay, let's see. The question is about wh...","[NUMBER]\n\nOkay, let's see. The question is a..."
716,716,Olmo-3-1025-7B,purity,I admire people who keep their virginity until...,"2\n\nOkay, let's see. The user is asking me to...","[NUMBER]\n\nOkay, let's see. The user is askin..."
717,717,Olmo-3-1025-7B,purity,I admire people who keep their virginity until...,"1\n\nOkay, let's see. The user is asking me to...","[NUMBER]\n\nOkay, let's see. The user is askin..."
718,718,Olmo-3-1025-7B,purity,I admire people who keep their virginity until...,"1\n\nOkay, let's see. The user wants me to rat...","[NUMBER]\n\nOkay, let's see. The user wants me..."


In [ ]:
df

,model,moral_value,statement,redacted_explanation
0,Qwen3-8B,authority,I believe that one of the most important value...,[NUMBER]\n\nI [EVALUATION] with the statement ...
1,Qwen3-8B,authority,I feel that most traditions serve a valuable f...,[NUMBER]\n\nI [EVALUATION] with the statement ...
2,Qwen3-8B,authority,I think having a strong leader is good for soc...,[NUMBER]\n\nI [EVALUATION] with the statement ...
3,Qwen3-8B,authority,I think it is important for societies to cheri...,[NUMBER]\n\nI [EVALUATION] with the statement ...
4,Qwen3-8B,authority,I think obedience to parents is an important v...,[NUMBER]\n\nI [EVALUATION] with the statement ...
5,Qwen3-8B,authority,We all need to learn from our elders.,[NUMBER]\n\nI [EVALUATION] with the statement ...
6,Qwen3-8B,care,Caring for people who have suffered is an impo...,Caring for people who have suffered is a funda...
7,Qwen3-8B,care,Everyone should try to comfort people who are ...,[NUMBER]\n\nI [EVALUATION] with the statement ...
8,Qwen3-8B,care,I am empathetic toward those people who have s...,[NUMBER]\n\nI [EVALUATION] with the statement ...
9,Qwen3-8B,care,I believe that compassion for those who are su...,[NUMBER] = Strongly [EVALUATION] \nI strongly...


In [ ]:
df.to_csv('annotation_sample_llama.csv', index=False)

In [ ]:
df = df[['index','model','moral_value','statement','sampled_explanations']]

KeyError: "['index', 'sampled_explanations'] not in index"

In [ ]:
df

,index,model,moral_value,statement,sampled_explanations
0,140,Qwen3-8B,authority,I believe that one of the most important value...,[NUMBER]\n\nI [EVALUATION] with the statement ...
1,125,Qwen3-8B,authority,I feel that most traditions serve a valuable f...,[NUMBER]\n\nI [EVALUATION] with the statement ...
2,147,Qwen3-8B,authority,I think having a strong leader is good for soc...,[NUMBER]\n\nI [EVALUATION] with the statement ...
3,121,Qwen3-8B,authority,I think it is important for societies to cheri...,[NUMBER]\n\nI [EVALUATION] with the statement ...
4,131,Qwen3-8B,authority,I think obedience to parents is an important v...,[NUMBER]\n\nI [EVALUATION] with the statement ...
5,136,Qwen3-8B,authority,We all need to learn from our elders.,[NUMBER]\n\nI [EVALUATION] with the statement ...
6,0,Qwen3-8B,care,Caring for people who have suffered is an impo...,Caring for people who have suffered is a funda...
7,24,Qwen3-8B,care,Everyone should try to comfort people who are ...,[NUMBER]\n\nI [EVALUATION] with the statement ...
8,15,Qwen3-8B,care,I am empathetic toward those people who have s...,[NUMBER]\n\nI [EVALUATION] with the statement ...
9,9,Qwen3-8B,care,I believe that compassion for those who are su...,[NUMBER] = Strongly [EVALUATION] \nI strongly...


In [5]:
import pandas as pd, numpy as np
from sklearn.metrics import cohen_kappa_score


marco_df = pd.read_csv('marco.csv').fillna(3)
marzia_df = pd.read_csv('marzia.csv').fillna(3)
rossana_df = pd.read_csv('rossana.csv').fillna(3)
shangrui_df = pd.read_csv('shangrui.csv').fillna(3)
florian_df = pd.read_csv('florian.csv').fillna(3)


marco = marco_df.score.to_numpy()
marzia = marzia_df.score.to_numpy()
rossana = rossana_df.score.to_numpy()
shangrui = shangrui_df.score.to_numpy()
florian = florian_df.score.to_numpy()

kappa_marco_marzia = cohen_kappa_score(marco, marzia)
kappa_marco_rossana = cohen_kappa_score(marco, rossana)
kappa_marco_shangrui = cohen_kappa_score(marco, shangrui)
kappa_marco_florian = cohen_kappa_score(marco, florian)
kappa_marzia_rossana = cohen_kappa_score(marzia, rossana)
kappa_marzia_shangrui = cohen_kappa_score(marzia, shangrui)
kappa_marzia_florian = cohen_kappa_score(marzia, florian)
kappa_rossana_shangrui = cohen_kappa_score(rossana, shangrui)
kappa_rossana_florian = cohen_kappa_score(rossana, florian)
kappa_shangrui_florian = cohen_kappa_score(shangrui, florian)

print("KAPPA COHEN SCORES:\n"
    f"kappa_marco_marzia={kappa_marco_marzia}\n kappa_marco_rossana={kappa_marco_rossana}\n "
    f"kappa_marco_shangrui={kappa_marco_shangrui}\n kappa_marco_florian={kappa_marco_florian}\n "
    f"kappa_marzia_rossana={kappa_marzia_rossana}\n kappa_marzia_shangrui={kappa_marzia_shangrui}\n "
    f"kappa_marzia_florian={kappa_marzia_florian}\n kappa_rossana_shangrui={kappa_rossana_shangrui}\n "
    f"kappa_rossana_florian={kappa_rossana_florian}\n kappa_shangrui_florian={kappa_shangrui_florian}"
)

accuracy_marco_marzia = np.mean(marco == marzia)
accuracy_marco_rossana = np.mean(marco == rossana)
accuracy_marco_shangrui = np.mean(marco == shangrui)
accuracy_marco_florian = np.mean(marco == florian)
accuracy_marzia_rossana = np.mean(marzia == rossana)
accuracy_marzia_shangrui = np.mean(marzia == shangrui)
accuracy_marzia_florian = np.mean(marzia == florian)
accuracy_rossana_shangrui = np.mean(rossana == shangrui)
accuracy_rossana_florian = np.mean(rossana == florian)
accuracy_shangrui_florian = np.mean(shangrui == florian)

print("ACCURACY:\n"
    f"accuracy_marco_marzia={accuracy_marco_marzia}\n accuracy_marco_rossana={accuracy_marco_rossana}\n "
    f"accuracy_marco_shangrui={accuracy_marco_shangrui}\n accuracy_marco_florian={accuracy_marco_florian}\n "
    f"accuracy_marzia_rossana={accuracy_marzia_rossana}\n accuracy_marzia_shangrui={accuracy_marzia_shangrui}\n "
    f"accuracy_marzia_florian={accuracy_marzia_florian}\n accuracy_rossana_shangrui={accuracy_rossana_shangrui}\n "
    f"accuracy_rossana_florian={accuracy_rossana_florian}\n accuracy_shangrui_florian={accuracy_shangrui_florian}"
)
marco_marzia = np.mean(abs(marco - marzia) )
marco_shangrui = np.mean(abs(marco - shangrui) )
marco_rossana = np.mean(abs(marco - rossana) )
marco_florian = np.mean(abs(marco-florian))
marzia_rossana = np.mean(abs(marzia - rossana) )
marzia_shangrui = np.mean(abs(marzia - shangrui) )
marzia_florian = np.mean(abs(marzia-florian))
rossana_shangrui = np.mean(abs(rossana - shangrui) )
rossana_florian = np.mean(abs(rossana-florian))
shangrui_florian = np.mean(abs(shangrui-florian))



print("MEAN ERROR:\n"
    f"marco_marzia={marco_marzia}\n marco_shangrui={marco_shangrui}\n "
    f"marco_rossana={marco_rossana}\n marco_florian={marco_florian}\n "
    f"marzia_rossana={marzia_rossana}\n marzia_shangrui={marzia_shangrui}\n "
    f"marzia_florian={marzia_florian}\n rossana_shangrui={rossana_shangrui}\n "
    f"rossana_florian={rossana_florian}\n shangrui_florian={shangrui_florian}"
)



KAPPA COHEN SCORES:
kappa_marco_marzia=0.5209899540948706
 kappa_marco_rossana=0.5702658708627237
 kappa_marco_shangrui=0.34514355528405616
 kappa_marco_florian=0.4367792838357273
 kappa_marzia_rossana=0.560528992878942
 kappa_marzia_shangrui=0.344578313253012
 kappa_marzia_florian=0.5156733485806538
 kappa_rossana_shangrui=0.4098360655737705
 kappa_rossana_florian=0.5777420931369126
 kappa_shangrui_florian=0.30343445511713074
ACCURACY:
accuracy_marco_marzia=0.6527777777777778
 accuracy_marco_rossana=0.6944444444444444
 accuracy_marco_shangrui=0.5347222222222222
 accuracy_marco_florian=0.5972222222222222
 accuracy_marzia_rossana=0.6875
 accuracy_marzia_shangrui=0.5277777777777778
 accuracy_marzia_florian=0.6527777777777778
 accuracy_rossana_shangrui=0.6041666666666666
 accuracy_rossana_florian=0.7083333333333334
 accuracy_shangrui_florian=0.5138888888888888
MEAN ERROR:
marco_marzia=0.3888888888888889
 marco_shangrui=0.4930555555555556
 marco_rossana=0.3194444444444444
 marco_florian=0.

,model,moral_value,statement,explanation,redacted_explanation,score
0,Falcon3-7B-Instruct,care,Caring for people who have suffered is an impo...,5 = Strongly agree\n\nCaring for people who ha...,\n\nCaring for people who have suffered is an ...,5
1,Falcon3-7B-Instruct,care,I believe that compassion for those who are su...,4 = Agree\n\nI agree with the statement becaus...,\n\n compassion is a fundamental aspect of hum...,5
2,Falcon3-7B-Instruct,care,We should all care for people who are in emoti...,4 = Agree\n\nI agree with the statement becaus...,\n\n showing empathy and compassion towards in...,5
3,Falcon3-7B-Instruct,care,I am empathetic toward those people who have s...,4 = Agree\n\nI agree with the statement becaus...,\n\n I believe empathy is a crucial aspect of ...,5
4,Falcon3-7B-Instruct,care,Everyone should try to comfort people who are ...,4 = Agree\n\nI agree with the statement becaus...,\n\n showing empathy and offering comfort to t...,5
...,...,...,...,...,...,...
139,Qwen3-8B,purity,I believe chastity is an important virtue.,"4\n\nI agree with the statement ""I believe cha...","[NUMBER]\n\n ""I believe chastity is an importa...",4
140,Qwen3-8B,purity,It upsets me when people use foul language lik...,4\n\nI agree with the statement because using ...,[NUMBER]\n\n using foul language in a dismissi...,4
141,Qwen3-8B,purity,If I found out that an acquaintance had an unu...,3\n\nI neither agree nor disagree because the ...,[NUMBER]\n\n because the reaction to someone's...,3
142,Qwen3-8B,purity,People should try to use natural medicines rat...,3\n\nI neither agree nor disagree with the sta...,[NUMBER]\n\n with the statement because the pr...,3


In [47]:
import pandas as pd, numpy as np
from sklearn.metrics import cohen_kappa_score


marco_df = pd.read_csv('marco.csv')
shangrui_df = pd.read_csv('shangrui.csv')


marco = marco_df.score.to_numpy()
shangrui = shangrui_df.score.to_numpy()

kappa_marco_shangrui = cohen_kappa_score(marco, shangrui)

print("KAPPA COHEN SCORES:\n"
    f"kappa_marco_shangrui={kappa_marco_shangrui}"
)

accuracy_marco_shangrui = np.mean(marco == shangrui)

print("ACCURACY:\n"
   
    f"accuracy_marco_shangrui={accuracy_marco_shangrui}"
)
marco_shangrui = np.mean(abs(marco - shangrui) )



print("GENERAL:\n"
    f" marco_shangrui={marco_shangrui}\n "
)



KAPPA COHEN SCORES:
kappa_marco_shangrui=0.34514355528405616
ACCURACY:
accuracy_marco_shangrui=0.5347222222222222
GENERAL:
 marco_shangrui=0.4930555555555556
 


In [48]:
models = marco_df.model.drop_duplicates().to_list()
for val in marco_df.moral_value.drop_duplicates().to_list():
    for model in models:
        marco = marco_df[(marco_df.moral_value==val)&(marco_df.model==model)].score.to_numpy()
        shangrui = shangrui_df[(shangrui_df.moral_value==val)&(shangrui_df.model==model)].score.to_numpy()
        accuracy_marco_shangrui = np.mean(marco == shangrui)

        print(f"{val.upper()} in {model.upper()}:\n"
            f"accuracy_marco_shangrui={accuracy_marco_shangrui}"
        )
        marco_shangrui = np.mean(abs(marco - shangrui) )



        print(f"{val.upper()} in {model.upper()}:\n"
            f"marco_shangrui={marco_shangrui}\n "
            
        )

CARE in FALCON3-7B-INSTRUCT:
accuracy_marco_shangrui=0.16666666666666666
CARE in FALCON3-7B-INSTRUCT:
marco_shangrui=1.0
 
CARE in OLMO-3-7B-INSTRUCT:
accuracy_marco_shangrui=1.0
CARE in OLMO-3-7B-INSTRUCT:
marco_shangrui=0.0
 
CARE in LLAMA-3.1-8B-INSTRUCT:
accuracy_marco_shangrui=0.5
CARE in LLAMA-3.1-8B-INSTRUCT:
marco_shangrui=0.5
 
CARE in QWEN3-8B:
accuracy_marco_shangrui=0.0
CARE in QWEN3-8B:
marco_shangrui=1.0
 
EQUALITY in FALCON3-7B-INSTRUCT:
accuracy_marco_shangrui=0.5
EQUALITY in FALCON3-7B-INSTRUCT:
marco_shangrui=0.5
 
EQUALITY in OLMO-3-7B-INSTRUCT:
accuracy_marco_shangrui=0.16666666666666666
EQUALITY in OLMO-3-7B-INSTRUCT:
marco_shangrui=0.8333333333333334
 
EQUALITY in LLAMA-3.1-8B-INSTRUCT:
accuracy_marco_shangrui=0.6666666666666666
EQUALITY in LLAMA-3.1-8B-INSTRUCT:
marco_shangrui=0.3333333333333333
 
EQUALITY in QWEN3-8B:
accuracy_marco_shangrui=0.3333333333333333
EQUALITY in QWEN3-8B:
marco_shangrui=0.6666666666666666
 
PROPORTIONALITY in FALCON3-7B-INSTRUCT:
accur

In [4]:
models = marco_df.model.drop_duplicates().to_list()
for model in models:
    marco = marco_df[(marco_df.model==model)].score.to_numpy()
    shangrui = shangrui_df[(shangrui_df.model==model)].score.to_numpy()
    accuracy_marco_shangrui = np.mean(marco == shangrui)

    print(f"{model.upper()}:\n"
        f"accuracy_marco_shangrui={accuracy_marco_shangrui}"
    )
    marco_shangrui = np.mean(abs(marco - shangrui) )



    print(f"{model.upper()}:\n"
        f"marco_shangrui={marco_shangrui}\n "
        
    )

FALCON3-7B-INSTRUCT:
accuracy_marco_shangrui=0.4444444444444444
FALCON3-7B-INSTRUCT:
marco_shangrui=0.6111111111111112
 
OLMO-3-7B-INSTRUCT:
accuracy_marco_shangrui=0.6666666666666666
OLMO-3-7B-INSTRUCT:
marco_shangrui=0.3333333333333333
 
LLAMA-3.1-8B-INSTRUCT:
accuracy_marco_shangrui=0.6111111111111112
LLAMA-3.1-8B-INSTRUCT:
marco_shangrui=0.3888888888888889
 
QWEN3-8B:
accuracy_marco_shangrui=0.4166666666666667
QWEN3-8B:
marco_shangrui=0.6388888888888888
 


### INTER MODEL ANALYSIS

### 1. models' variability

In [7]:
import pandas as pd

df = pd.read_csv('marco.csv')
shangrui = pd.read_csv('shangrui.csv')
marzia = pd.read_csv('marzia.csv').fillna(3)
rossana = pd.read_csv('rossana.csv').fillna(3)
florian = pd.read_csv('florian.csv').fillna(3)
df['score'] = (df.score + shangrui.score+marzia.score+rossana.score+florian.score)/5

df_int = pd.read_csv('questionnaire_scores.csv')

In [ ]:
df

,model,moral_value,statement,explanation,redacted_explanation,score
0,Falcon3-7B-Instruct,care,Caring for people who have suffered is an impo...,5 = Strongly agree\n\nCaring for people who ha...,\n\nCaring for people who have suffered is an ...,5.0
1,Falcon3-7B-Instruct,care,I believe that compassion for those who are su...,4 = Agree\n\nI agree with the statement becaus...,\n\n compassion is a fundamental aspect of hum...,4.8
2,Falcon3-7B-Instruct,care,We should all care for people who are in emoti...,4 = Agree\n\nI agree with the statement becaus...,\n\n showing empathy and compassion towards in...,4.8
3,Falcon3-7B-Instruct,care,I am empathetic toward those people who have s...,4 = Agree\n\nI agree with the statement becaus...,\n\n I believe empathy is a crucial aspect of ...,4.8
4,Falcon3-7B-Instruct,care,Everyone should try to comfort people who are ...,4 = Agree\n\nI agree with the statement becaus...,\n\n showing empathy and offering comfort to t...,4.0
...,...,...,...,...,...,...
139,Qwen3-8B,purity,I believe chastity is an important virtue.,"4\n\nI agree with the statement ""I believe cha...","[NUMBER]\n\n ""I believe chastity is an importa...",4.4
140,Qwen3-8B,purity,It upsets me when people use foul language lik...,4\n\nI agree with the statement because using ...,[NUMBER]\n\n using foul language in a dismissi...,4.6
141,Qwen3-8B,purity,If I found out that an acquaintance had an unu...,3\n\nI neither agree nor disagree because the ...,[NUMBER]\n\n because the reaction to someone's...,3.0
142,Qwen3-8B,purity,People should try to use natural medicines rat...,3\n\nI neither agree nor disagree with the sta...,[NUMBER]\n\n with the statement because the pr...,2.8


In [9]:
df_int

,model,moral_value,statement,qst_score
0,Falcon3-7B-Instruct,care,Caring for people who have suffered is an impo...,5
1,Falcon3-7B-Instruct,care,I believe that compassion for those who are su...,5
2,Falcon3-7B-Instruct,care,We should all care for people who are in emoti...,5
3,Falcon3-7B-Instruct,care,I am empathetic toward those people who have s...,4
4,Falcon3-7B-Instruct,care,Everyone should try to comfort people who are ...,4
...,...,...,...,...
139,Qwen3-8B,purity,I believe chastity is an important virtue.,3
140,Qwen3-8B,purity,It upsets me when people use foul language lik...,4
141,Qwen3-8B,purity,If I found out that an acquaintance had an unu...,3
142,Qwen3-8B,purity,People should try to use natural medicines rat...,3


In [20]:
df['qst_score'] = df_int.qst_score

In [22]:
moral_val = df.groupby(['model', 'moral_value']).aggregate({'score':'mean', 'qst_score':'mean'}).reset_index()

In [23]:
moral_val.to_csv('model_moral_values.csv', index=False)

In [24]:
moral_val

,model,moral_value,score,qst_score
0,Falcon3-7B-Instruct,authority,3.866667,3.666667
1,Falcon3-7B-Instruct,care,4.666667,4.666667
2,Falcon3-7B-Instruct,equality,2.966667,3.333333
3,Falcon3-7B-Instruct,loyalty,4.266667,4.000000
4,Falcon3-7B-Instruct,proportionality,3.733333,4.000000
5,Falcon3-7B-Instruct,purity,3.400000,3.500000
6,Llama-3.1-8B-Instruct,authority,4.366667,4.166667
7,Llama-3.1-8B-Instruct,care,4.866667,5.000000
8,Llama-3.1-8B-Instruct,equality,3.700000,3.333333
9,Llama-3.1-8B-Instruct,loyalty,4.533333,4.666667


In [25]:
general_var = df.groupby(['model']).score.apply(list).reset_index()

In [26]:
general_var_int = df_int.groupby(['model']).qst_score.apply(list).reset_index()

In [12]:
general_var

,model,score
0,Falcon3-7B-Instruct,"[5.0, 4.8, 4.8, 4.8, 4.0, 4.6, 2.4, 3.4, 1.8, ..."
1,Llama-3.1-8B-Instruct,"[5.0, 5.0, 4.8, 4.8, 4.8, 4.8, 4.0, 4.8, 2.4, ..."
2,Olmo-3-7B-Instruct,"[5.0, 5.0, 5.0, 4.4, 4.4, 4.2, 2.4, 2.8, 2.4, ..."
3,Qwen3-8B,"[4.8, 4.8, 4.8, 4.8, 4.2, 4.8, 2.8, 1.8, 1.6, ..."


In [33]:
from scipy.stats import wilcoxon
from itertools import combinations
from scipy.stats import kruskal

general_var
# Extract models and their scores
kruskall_results = kruskal(general_var.score[0], general_var.score[1], general_var.score[2], general_var.score[3])

kruskall_results

KruskalResult(statistic=11.205406706810223, pvalue=0.010665468590207547)

In [28]:
from scipy.stats import wilcoxon
from itertools import combinations
from scipy.stats import kruskal


# Extract models and their scores
kruskall_results = kruskal(general_var_int.qst_score[0], general_var_int.qst_score[1], general_var_int.qst_score[2], general_var_int.qst_score[3])

kruskall_results

KruskalResult(statistic=14.622400998448336, pvalue=0.0021694903685278845)

In [34]:
from scipy.stats import wilcoxon
from itertools import combinations
from scipy.stats import false_discovery_control

# Extract models and their scores
models_scores = general_var.set_index('model')['score'].to_dict()

# Compute pairwise Wilcoxon signed-rank tests
results = {}
for model1, model2 in combinations(models_scores.keys(), 2):
    try:
        stat, p_value = wilcoxon(models_scores[model1], models_scores[model2])
        results[f"{model1} vs {model2}"] = {'statistic': stat, 'p_value': p_value}
    except ValueError as e:
        results[f"{model1} vs {model2}"] = {'error': str(e)}

# Print results
for pair, result in results.items():
    print(f"{pair}: {result}")

Falcon3-7B-Instruct vs Llama-3.1-8B-Instruct: {'statistic': 40.5, 'p_value': 0.0005802604739283961}
Falcon3-7B-Instruct vs Olmo-3-7B-Instruct: {'statistic': 237.5, 'p_value': 0.6196957050787304}
Falcon3-7B-Instruct vs Qwen3-8B: {'statistic': 170.0, 'p_value': 0.30191553586724107}
Llama-3.1-8B-Instruct vs Olmo-3-7B-Instruct: {'statistic': 63.5, 'p_value': 0.0002961968567425577}
Llama-3.1-8B-Instruct vs Qwen3-8B: {'statistic': 32.5, 'p_value': 6.22702903679871e-05}
Olmo-3-7B-Instruct vs Qwen3-8B: {'statistic': 231.0, 'p_value': 0.536031291947968}


In [32]:
from scipy.stats import wilcoxon
from itertools import combinations
from scipy.stats import false_discovery_control

# Extract models and their scores
models_scores = general_var_int.set_index('model')['qst_score'].to_dict()

# Compute pairwise Wilcoxon signed-rank tests
results = {}
for model1, model2 in combinations(models_scores.keys(), 2):
    try:
        stat, p_value = wilcoxon(models_scores[model1], models_scores[model2])
        results[f"{model1} vs {model2}"] = {'statistic': stat, 'p_value': p_value}
    except ValueError as e:
        results[f"{model1} vs {model2}"] = {'error': str(e)}

# Print results
for pair, result in results.items():
    print(f"{pair}: {result}")

Falcon3-7B-Instruct vs Llama-3.1-8B-Instruct: {'statistic': 8.0, 'p_value': 0.0007891129890156302}
Falcon3-7B-Instruct vs Olmo-3-7B-Instruct: {'statistic': 70.0, 'p_value': 0.25134910881022265}
Falcon3-7B-Instruct vs Qwen3-8B: {'statistic': 5.5, 'p_value': 0.007526315166457873}
Llama-3.1-8B-Instruct vs Olmo-3-7B-Instruct: {'statistic': 15.0, 'p_value': 0.0011341081440840795}
Llama-3.1-8B-Instruct vs Qwen3-8B: {'statistic': 0.0, 'p_value': 4.80954741285457e-05}
Olmo-3-7B-Instruct vs Qwen3-8B: {'statistic': 54.0, 'p_value': 0.22525290636065332}


In [36]:
print(results)

{'Falcon3-7B-Instruct vs Llama-3.1-8B-Instruct': {'statistic': 40.5, 'p_value': 0.0005802604739283961}, 'Falcon3-7B-Instruct vs Olmo-3-7B-Instruct': {'statistic': 237.5, 'p_value': 0.6196957050787304}, 'Falcon3-7B-Instruct vs Qwen3-8B': {'statistic': 170.0, 'p_value': 0.30191553586724107}, 'Llama-3.1-8B-Instruct vs Olmo-3-7B-Instruct': {'statistic': 63.5, 'p_value': 0.0002961968567425577}, 'Llama-3.1-8B-Instruct vs Qwen3-8B': {'statistic': 32.5, 'p_value': 6.22702903679871e-05}, 'Olmo-3-7B-Instruct vs Qwen3-8B': {'statistic': 231.0, 'p_value': 0.536031291947968}}


In [37]:
from scipy.stats import false_discovery_control

vals = [x['p_value'] for x in results.values() if 'p_value' in x]
print(vals)
fls = false_discovery_control(vals)
print(fls)

[0.0005802604739283961, 0.6196957050787304, 0.30191553586724107, 0.0002961968567425577, 6.22702903679871e-05, 0.536031291947968]
[1.16052095e-03 6.19695705e-01 4.52873304e-01 8.88590570e-04
 3.73621742e-04 6.19695705e-01]


In [17]:

# Extract models and their scores
models_scores = general_var.set_index('model')['score'].to_dict()

# Compute pairwise Wilcoxon signed-rank tests
results = {}
for model1, model2 in combinations(models_scores.keys(), 2):
    try:
        stat, p_value = wilcoxon(models_scores[model1], models_scores[model2])
        results[f"{model1} vs {model2}"] = {'statistic': stat, 'p_value': p_value}
    except ValueError as e:
        results[f"{model1} vs {model2}"] = {'error': str(e)}

# Print results
for pair, result in results.items():
    print(f"{pair}: {result}")

Falcon3-7B-Instruct vs Llama-3.1-8B-Instruct: {'statistic': 40.5, 'p_value': 0.0005802604739283961}
Falcon3-7B-Instruct vs Olmo-3-7B-Instruct: {'statistic': 237.5, 'p_value': 0.6196957050787304}
Falcon3-7B-Instruct vs Qwen3-8B: {'statistic': 170.0, 'p_value': 0.30191553586724107}
Llama-3.1-8B-Instruct vs Olmo-3-7B-Instruct: {'statistic': 63.5, 'p_value': 0.0002961968567425577}
Llama-3.1-8B-Instruct vs Qwen3-8B: {'statistic': 32.5, 'p_value': 6.22702903679871e-05}
Olmo-3-7B-Instruct vs Qwen3-8B: {'statistic': 231.0, 'p_value': 0.536031291947968}


In [18]:
from scipy.stats import false_discovery_control

vals = [x['p_value'] for x in results.values() if 'p_value' in x]

fls = false_discovery_control(vals)
print(fls)

[1.16052095e-03 6.19695705e-01 4.52873304e-01 8.88590570e-04
 3.73621742e-04 6.19695705e-01]


### CHECK SCORES AND HUMAN EVALUATIONS

In [38]:
int_ext = df.merge(df_int)

int_ext = int_ext.groupby(['model', 'moral_value']).aggregate({'score': 'mean', 'qst_score': 'mean'}).reset_index()

In [39]:
int_ext

,model,moral_value,score,qst_score
0,Falcon3-7B-Instruct,authority,3.866667,3.666667
1,Falcon3-7B-Instruct,care,4.666667,4.666667
2,Falcon3-7B-Instruct,equality,2.966667,3.333333
3,Falcon3-7B-Instruct,loyalty,4.266667,4.000000
4,Falcon3-7B-Instruct,proportionality,3.733333,4.000000
5,Falcon3-7B-Instruct,purity,3.400000,3.500000
6,Llama-3.1-8B-Instruct,authority,4.366667,4.166667
7,Llama-3.1-8B-Instruct,care,4.866667,5.000000
8,Llama-3.1-8B-Instruct,equality,3.700000,3.333333
9,Llama-3.1-8B-Instruct,loyalty,4.533333,4.666667


### CHECK model and human profiles

In [40]:
import pandas as pd
humans = pd.read_csv('data/d3-raters.csv')

humans

,rater_id,Age,Gender,Region,Country,SES,care,equality,proportionality,authority,loyalty,purity
0,R_2dzroRQ2uSeR5kh,18-30,Man,Arab Culture,Egypt,5,3.833333,3.000000,4.166667,4.000000,4.666667,2.833333
1,R_3sKDIHxmcW6hfUd,18-30,Man,Arab Culture,Qatar,6,4.666667,3.833333,4.666667,4.500000,5.000000,4.333333
2,R_3nIyQhjRYe275AC,30-50,Man,Arab Culture,United Arab Emirates,5,4.833333,2.000000,4.833333,4.833333,4.666667,3.166667
3,R_3FKXkB2A9MBDAmc,30-50,Man,Arab Culture,United Arab Emirates,3,5.000000,3.166667,4.000000,3.500000,4.333333,1.000000
4,R_bx6rt0lKsUTORvX,30-50,Man,Arab Culture,United Arab Emirates,7,3.666667,3.000000,4.166667,4.166667,3.666667,3.666667
...,...,...,...,...,...,...,...,...,...,...,...,...
4304,R_02Kt90m9vxg3qx3,18-30,Other,Sub Saharan Africa,Nigeria,5,4.833333,1.000000,4.500000,2.333333,3.666667,1.666667
4305,R_1F9ue1HaVucwtWR,30-50,Man,Western Europe,United Kingdom,8,4.333333,1.833333,3.666667,4.500000,2.666667,2.166667
4306,R_2qrau3NjTbfBA87,30-50,Man,Western Europe,United Kingdom,7,3.833333,2.500000,3.833333,4.166667,3.500000,2.333333
4307,R_2f1CnqHV5qoegV4,30-50,Woman,Western Europe,United Kingdom,5,3.833333,3.166667,3.333333,4.000000,3.666667,2.666667


In [41]:
from sklearn.metrics.pairwise import cosine_similarity

int_ext_pivot = int_ext.pivot(index='model', columns='moral_value', values='qst_score').reset_index()

# Extract moral values columns for models and humans
model_vectors = int_ext_pivot[['authority', 'care', 'equality', 'loyalty', 'proportionality', 'purity']].values
human_vectors = humans[['care', 'equality', 'proportionality', 'authority', 'loyalty', 'purity']].values

# Compute cosine similarities for each rater with each model
cosine_similarities = cosine_similarity(human_vectors, model_vectors)

# Add cosine similarities for each model as columns in the humans dataframe
for i, model_name in enumerate(int_ext_pivot['model']):
    humans[model_name + '_cosine_similarity'] = cosine_similarities[:, i]

In [42]:
for model in ['Falcon3-7B-Instruct_cosine_similarity', 'Llama-3.1-8B-Instruct_cosine_similarity', 'Olmo-3-7B-Instruct_cosine_similarity', 'Qwen3-8B_cosine_similarity']:
    humans[f'{model}_quantile'] = pd.qcut(humans[model], 10, labels=False) + 1

In [43]:
# Check if all quantile columns have the same value for each human
same_quantile_humans = humans[
    (humans['Falcon3-7B-Instruct_cosine_similarity_quantile'] == humans['Llama-3.1-8B-Instruct_cosine_similarity_quantile']) &
    (humans['Falcon3-7B-Instruct_cosine_similarity_quantile'] == humans['Olmo-3-7B-Instruct_cosine_similarity_quantile']) &
    (humans['Falcon3-7B-Instruct_cosine_similarity_quantile'] == humans['Qwen3-8B_cosine_similarity_quantile'])
]

# Count the number of such humans
num_same_quantile_humans = len(same_quantile_humans)
print(f"Number of humans always on the same quantile: {num_same_quantile_humans}")

Number of humans always on the same quantile: 1853


In [44]:
# Find humans in different quantiles
different_quantile_humans = humans[
    (humans['Falcon3-7B-Instruct_cosine_similarity_quantile'] != humans['Llama-3.1-8B-Instruct_cosine_similarity_quantile']) |
    (humans['Falcon3-7B-Instruct_cosine_similarity_quantile'] != humans['Olmo-3-7B-Instruct_cosine_similarity_quantile']) |
    (humans['Falcon3-7B-Instruct_cosine_similarity_quantile'] != humans['Qwen3-8B_cosine_similarity_quantile'])
]

# Display the result
different_quantile_humans

,rater_id,Age,Gender,Region,Country,SES,care,equality,proportionality,authority,loyalty,purity,Falcon3-7B-Instruct_cosine_similarity,Llama-3.1-8B-Instruct_cosine_similarity,Olmo-3-7B-Instruct_cosine_similarity,Qwen3-8B_cosine_similarity,Falcon3-7B-Instruct_cosine_similarity_quantile,Llama-3.1-8B-Instruct_cosine_similarity_quantile,Olmo-3-7B-Instruct_cosine_similarity_quantile,Qwen3-8B_cosine_similarity_quantile
0,R_2dzroRQ2uSeR5kh,18-30,Man,Arab Culture,Egypt,5,3.833333,3.000000,4.166667,4.000000,4.666667,2.833333,0.975481,0.973577,0.969802,0.974764,7,7,6,6
5,R_1o1h2Jdpj61CxBY,18-30,Man,Arab Culture,Egypt,5,4.500000,3.666667,5.000000,5.000000,4.666667,4.666667,0.980581,0.979363,0.973268,0.978896,8,8,7,7
7,R_3NRsIIcMpRpGaJb,18-30,Man,Arab Culture,United Arab Emirates,5,5.000000,1.833333,4.833333,4.500000,4.833333,1.833333,0.917485,0.917907,0.913149,0.920156,1,2,1,2
11,R_110jt65DHeI6mCQ,50+,Man,Indian Cultural Sphere,India,7,2.666667,3.333333,3.333333,3.500000,3.333333,3.166667,0.993165,0.990633,0.984647,0.989974,10,10,9,9
13,R_2SJ1pd4iukv75UO,30-50,Man,Indian Cultural Sphere,India,6,4.333333,4.500000,4.666667,4.333333,5.000000,4.666667,0.991813,0.988726,0.988498,0.990161,10,9,9,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4298,R_237CqI0Jr9kqPb4,30-50,Woman,Oceania,Australia,6,2.833333,1.666667,2.833333,3.500000,3.166667,1.833333,0.956270,0.958750,0.944898,0.956096,4,5,3,4
4302,R_NWNb0dIU6GJ3hxT,30-50,Man,Sub Saharan Africa,Ghana,8,4.666667,1.166667,3.333333,4.500000,3.833333,3.666667,0.922716,0.928978,0.915258,0.925034,2,2,1,2
4305,R_1F9ue1HaVucwtWR,30-50,Man,Western Europe,United Kingdom,8,4.333333,1.833333,3.666667,4.500000,2.666667,2.166667,0.931546,0.936458,0.923967,0.934974,2,3,2,2
4306,R_2qrau3NjTbfBA87,30-50,Man,Western Europe,United Kingdom,7,3.833333,2.500000,3.833333,4.166667,3.500000,2.333333,0.966384,0.967325,0.959053,0.967243,6,6,5,5


In [24]:
humans

,rater_id,Age,Gender,Region,Country,SES,care,equality,proportionality,authority,loyalty,purity,Falcon3-7B-Instruct_cosine_similarity,Llama-3.1-8B-Instruct_cosine_similarity,Olmo-3-7B-Instruct_cosine_similarity,Qwen3-8B_cosine_similarity,Falcon3-7B-Instruct_cosine_similarity_quantile,Llama-3.1-8B-Instruct_cosine_similarity_quantile,Olmo-3-7B-Instruct_cosine_similarity_quantile,Qwen3-8B_cosine_similarity_quantile
0,R_2dzroRQ2uSeR5kh,18-30,Man,Arab Culture,Egypt,5,3.833333,3.000000,4.166667,4.000000,4.666667,2.833333,0.975481,0.973577,0.969802,0.974764,7,7,6,6
1,R_3sKDIHxmcW6hfUd,18-30,Man,Arab Culture,Qatar,6,4.666667,3.833333,4.666667,4.500000,5.000000,4.333333,0.985539,0.983899,0.981943,0.984960,8,8,8,8
2,R_3nIyQhjRYe275AC,30-50,Man,Arab Culture,United Arab Emirates,5,4.833333,2.000000,4.833333,4.833333,4.666667,3.166667,0.941927,0.942873,0.934664,0.942704,3,3,3,3
3,R_3FKXkB2A9MBDAmc,30-50,Man,Arab Culture,United Arab Emirates,3,5.000000,3.166667,4.000000,3.500000,4.333333,1.000000,0.937051,0.935999,0.942431,0.942710,3,3,3,3
4,R_bx6rt0lKsUTORvX,30-50,Man,Arab Culture,United Arab Emirates,7,3.666667,3.000000,4.166667,4.166667,3.666667,3.666667,0.979848,0.978650,0.972029,0.978216,7,7,7,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4304,R_02Kt90m9vxg3qx3,18-30,Other,Sub Saharan Africa,Nigeria,5,4.833333,1.000000,4.500000,2.333333,3.666667,1.666667,0.867570,0.863562,0.875949,0.873117,1,1,1,1
4305,R_1F9ue1HaVucwtWR,30-50,Man,Western Europe,United Kingdom,8,4.333333,1.833333,3.666667,4.500000,2.666667,2.166667,0.931546,0.936458,0.923967,0.934974,2,3,2,2
4306,R_2qrau3NjTbfBA87,30-50,Man,Western Europe,United Kingdom,7,3.833333,2.500000,3.833333,4.166667,3.500000,2.333333,0.966384,0.967325,0.959053,0.967243,6,6,5,5
4307,R_2f1CnqHV5qoegV4,30-50,Woman,Western Europe,United Kingdom,5,3.833333,3.166667,3.333333,4.000000,3.666667,2.666667,0.987252,0.988501,0.982523,0.988357,9,9,8,9


In [45]:
mod

NameError: name 'mod' is not defined

In [49]:
ratings = pd.read_csv('data/d3-ratings.csv')

for model in models:
    mod = pd.read_csv(f'output/model_ratings/{model}_ratings.csv')
    hm = humans[(humans[model + '_cosine_similarity_quantile'] == 10)][['rater_id']].merge(ratings).merge(mod)
    hm.to_csv(f'output/offensive_analysis/{model}_quantile_10_humans.csv', index=False)
    
    

In [26]:
ratings = pd.read_csv('data/d3-ratings.csv')

for model in models:
    mod = pd.read_csv(f'output/model_ratings/{model}_ratings.csv')
    hm = humans.merge(ratings).merge(mod)
    hm.to_csv(f'output/offensive_analysis/{model}_humans.csv', index=False)
    
    

In [51]:
ratings = pd.read_csv('data/d3-ratings.csv')

for model in models:
    mod = pd.read_csv(f'output/model_ratings/{model}_ratings.csv')
    hm = humans[(humans[model + '_cosine_similarity_quantile'] == 10)][['rater_id']].merge(ratings).merge(mod)
    hm.to_csv(f'output/offensive_analysis/{model}_quantile_10_humans.csv', index=False)
    mod = pd.read_csv(f'output/model_ratings/{model}_ratings.csv')
    hm = humans[(humans[model + '_cosine_similarity_quantile'] == 1)][['rater_id']].merge(ratings).merge(mod)
    hm.to_csv(f'output/offensive_analysis/{model}_quantile_1_humans.csv', index=False)
    
    

In [59]:
import os
import pandas as pd
from sklearn.metrics import f1_score

# Create the output directory if it doesn't exist
output_folder = 'output/offensive_f1'
os.makedirs(output_folder, exist_ok=True)

# Iterate through each file in the folder
input_folder = 'output/offensive_analysis'
for file in os.listdir(input_folder):
    if file.endswith('.csv'):
        file_path = os.path.join(input_folder, file)
        
        # Read the CSV file
        df = pd.read_csv(file_path)
        
        # Remove rows where rating_raw == -1
        df = df[df['rating_raw'] != -1]
        
        # Compute F1-score for the model with each rater
        raters = df['rater_id'].unique()
        f1_scores = []
        for rater in raters:
            rater_data = df[df['rater_id'] == rater]
            if len(rater_data) > 1:  # Ensure there are enough data points
                f1 = f1_score(rater_data['rating_raw'], rater_data['model_rating'], average='weighted')
                f1_scores.append({'rater_id': rater, 'f1_score': f1})
        
        # Save the results to a new CSV file
        output_file_path = os.path.join(output_folder, file)
        pd.DataFrame(f1_scores).to_csv(output_file_path, index=False)

In [52]:
from sklearn.metrics import mean_squared_error,mean_absolute_error
import os,pandas as pd
# Create the output directory if it doesn't exist
output_folder = 'output/offensive_mse'
os.makedirs(output_folder, exist_ok=True)

# Iterate through each file in the folder
input_folder = 'output/offensive_analysis'
for file in os.listdir(input_folder):
    if file.endswith('.csv'):
        file_path = os.path.join(input_folder, file)
        
        # Read the CSV file
        df = pd.read_csv(file_path)
        
        # Remove rows where rating_raw == -1
        df = df[df['rating_raw'] != -1]
        df['delta'] = df.model_score - df.rating_raw
        # Compute mean-squared error for the model with each rater
        raters = df['rater_id'].unique()
        mse_scores = []
        for rater in raters:
            rater_data = df[df['rater_id'] == rater]
            if len(rater_data) > 1:  # Ensure there are enough data points
                
                mse = mean_absolute_error(rater_data['rating_raw'], rater_data['model_score'])
                mse_polarity = rater_data.model_score - rater_data.rating_raw
                
                mse_scores.append({'rater_id': rater, 'mse_score': mse,'mse_polarity': rater_data.delta.mean()})
        
        # Save the results to a new CSV file
        output_file_path = os.path.join(output_folder, file)
        pd.DataFrame(mse_scores).to_csv(output_file_path, index=False)

In [50]:
import os
import pandas as pd

# Define the folder containing the files
folder = 'output/offensive_mse'

# Iterate through each file in the folder
for file in os.listdir(folder):
    if file.endswith('.csv'):
        # Read the file
        file_path = os.path.join(folder, file)
        df = pd.read_csv(file_path)
        
        # Calculate the average MSE
        avg_mse = df['mse_score'].mean()
        avg_polarity = df['mse_polarity'].mean()
        
        # Print the file name and average MSE
        print(f"File: {file}, Average MSE: {avg_mse}, Average Polarity: {avg_polarity}")


File: Qwen3-8B_humans.csv, Average MSE: 1.845667366551316, Average Polarity: 1.583363034396558
File: Llama-3.1-8B-Instruct_humans.csv, Average MSE: 1.934877486587593, Average Polarity: 1.6969504742339252
File: Olmo-3-7B-Instruct_humans.csv, Average MSE: 2.4725011482371806, Average Polarity: 2.3848176736208124
File: Falcon3-7B-Instruct_humans.csv, Average MSE: 1.4771588407534784, Average Polarity: 1.0488286597529293


In [56]:
import os
import pandas as pd

# Define the folder containing the files
folder = 'output/offensive_mse'

# Iterate through each file in the folder
for file in os.listdir(folder):
    if 'quantile' in file and file.endswith('.csv'):
        # Read the file
        file_path = os.path.join(folder, file)
        df = pd.read_csv(file_path)
        
        # Calculate the average MSE
        avg_mse = df['mse_score'].mean()
        avg_polarity = df['mse_polarity'].mean()
        
        # Print the file name and average MSE
        print(f"File: {file}, Average MSE: {avg_mse}, Average Polarity: {avg_polarity}")


File: Qwen3-8B_quantile_1_humans.csv, Average MSE: 1.9007118447475826, Average Polarity: 1.6774733205538865
File: Falcon3-7B-Instruct_quantile_10_humans.csv, Average MSE: 1.6365119783546784, Average Polarity: 1.126275721921771
File: Falcon3-7B-Instruct_quantile_1_humans.csv, Average MSE: 1.5000336645008345, Average Polarity: 1.11937895479614
File: Olmo-3-7B-Instruct_quantile_10_humans.csv, Average MSE: 2.580470952163602, Average Polarity: 2.4697392619608323
File: Llama-3.1-8B-Instruct_quantile_10_humans.csv, Average MSE: 2.0671182799491987, Average Polarity: 1.773201360785709
File: Qwen3-8B_quantile_10_humans.csv, Average MSE: 1.9785441703146196, Average Polarity: 1.6583165101685549
File: Llama-3.1-8B-Instruct_quantile_1_humans.csv, Average MSE: 1.9544370900897243, Average Polarity: 1.7555201970729115
File: Olmo-3-7B-Instruct_quantile_1_humans.csv, Average MSE: 2.5600073295054573, Average Polarity: 2.475622489393146


In [88]:
import os
import glob
import pandas as pd
from itertools import combinations
from sklearn.metrics import mean_absolute_error

# Define the folder containing the model ratings
files = [x for x in glob.glob('output/model_ratings/*.csv')]
pairs = list(combinations(files, 2))


# Read all model rating files

for pair in pairs:
    df1 = pd.read_csv(pair[0])
    df2 = pd.read_csv(pair[1])
    
    
    mse = mean_absolute_error(df1.model_score, df2.model_score)
    
    print(f"Comparison: {os.path.basename(pair[0])} vs {os.path.basename(pair[1])}, MSE: {mse}")


Comparison: Olmo-3-7B-Instruct_ratings.csv vs Falcon3-7B-Instruct_ratings.csv, MSE: 1.3658379768538362
Comparison: Olmo-3-7B-Instruct_ratings.csv vs Llama-3.1-8B-Instruct_ratings.csv, MSE: 0.8656236605229318
Comparison: Olmo-3-7B-Instruct_ratings.csv vs Qwen3-8B_ratings.csv, MSE: 0.9828546935276468
Comparison: Falcon3-7B-Instruct_ratings.csv vs Llama-3.1-8B-Instruct_ratings.csv, MSE: 0.8469781397342477
Comparison: Falcon3-7B-Instruct_ratings.csv vs Qwen3-8B_ratings.csv, MSE: 0.6885983711958851
Comparison: Llama-3.1-8B-Instruct_ratings.csv vs Qwen3-8B_ratings.csv, MSE: 0.6105872267466781


### CAUSAL GRAPH

In [55]:
raters = pd.read_csv('data/d3-raters.csv')

prova = pd.read_csv('output/offensive_mse/Olmo-3-7B-Instruct_humans.csv')

df = prova.merge(raters, on='rater_id')

df = df[df.Age!='50+']
df = df[df.Gender!='Other']

-0.044984856033999154

0.021081441488231878

In [73]:
import numpy as np
from itertools import combinations

l = list()
for item in df.Age.drop_duplicates().to_list():
    print(np.std(df[df.Age==item].mse_score))
    l.append(np.mean(df[df.Age==item].mse_score))

print(f"Age: {l[0]-l[1]}")

l = list()
for item in df.Gender.drop_duplicates().to_list():
    print(np.std(df[df.Gender==item].mse_score))
    l.append(np.mean(df[df.Gender==item].mse_score))

l[0]-l[1]
print(f"Gender: {l[0]-l[1]}")

# Compute mean mse_score per Region
means = df.groupby("Region")["mse_score"].mean()

stds = df.groupby("Region")["mse_score"].std()

# Generate all unique unordered pairs and compute differences
for (region1, value1), (region2, value2) in combinations(means.items(), 2):
    diff = value1 - value2
    print(f"{region1} vs {region2} → Difference: {diff}")

print(stds)


0.33359970299304975
0.3732470854301126
Age: -0.044984856033999154
0.3523780023712248
0.3511478609915833
Gender: 0.021081441488231878
Arab Culture vs Indian Cultural Sphere → Difference: -0.014562463845761098
Arab Culture vs Latin America → Difference: 0.03786755531336805
Arab Culture vs North America → Difference: -0.04696827965786454
Arab Culture vs Oceania → Difference: -0.06567531185468045
Arab Culture vs Sinosphere → Difference: -0.11050926303664355
Arab Culture vs Sub Saharan Africa → Difference: -0.026192683762877866
Arab Culture vs Western Europe → Difference: 0.001236747844729269
Indian Cultural Sphere vs Latin America → Difference: 0.05243001915912915
Indian Cultural Sphere vs North America → Difference: -0.032405815812103445
Indian Cultural Sphere vs Oceania → Difference: -0.051112848008919354
Indian Cultural Sphere vs Sinosphere → Difference: -0.09594679919088245
Indian Cultural Sphere vs Sub Saharan Africa → Difference: -0.011630219917116769
Indian Cultural Sphere vs Wester

In [65]:
import numpy as np
from itertools import combinations

def best_worst_pair(df, col, value_col="mse_score"):
    means = df.groupby(col)[value_col].mean()
    
    best_group = means.idxmin()   # lowest mse = best
    worst_group = means.idxmax()  # highest mse = worst
    
    best_val = means.loc[best_group]
    worst_val = means.loc[worst_group]
    
    diff = worst_val - best_val  # always positive
    
    return (best_group, worst_group, best_val, worst_val), diff

def greedy_best_worst(df, columns, value_col="mse_score"):
    current_df = df.copy()
    history = []
    remaining_cols = columns.copy()
    
    while remaining_cols:
        best_col = None
        best_pair_vals = None
        best_diff = -np.inf
        
        # Find column with largest best-vs-worst gap
        for col in remaining_cols:
            pair, diff = best_worst_pair(current_df, col, value_col)
            
            if diff > best_diff:
                best_diff = diff
                best_col = col
                best_pair_vals = pair
        
        if best_col is None:
            break
        
        best_g, worst_g, best_v, worst_v = best_pair_vals
        
        # Store full info
        history.append({
            "column": best_col,
            "best_group": best_g,
            "worst_group": worst_g,
            "best_mean": best_v,
            "worst_mean": worst_v,
            "diff": best_diff,
            "remaining_rows": len(current_df)
        })
        
        # 🔥 Reduce dataset to ONLY best + worst groups
        current_df = current_df[current_df[best_col].isin([best_g, worst_g])]
        
        remaining_cols.remove(best_col)
    
    return history

In [ ]:
cols = ["Age", "Gender", "Region"]

raters = pd.read_csv('data/d3-raters.csv')

prova = pd.read_csv('output/offensive_mse/Falcon3-7B-Instruct_humans.csv')

df = prova.merge(raters, on='rater_id')


df = df[df.Age!='50+']
df = df[df.Gender!='Other']

history = greedy_best_worst(df, cols)

for i, h in enumerate(history, 1):
    print(f"Step {i}")
    print(f" Column: {h['column']}")
    print(f" Best: {h['best_group']} ({h['best_mean']})")
    print(f" Worst: {h['worst_group']} ({h['worst_mean']})")
    print(f" Gap: {h['diff']}")
    print(f" Remaining rows: {h['remaining_rows']}")
    print()

4309
Step 1
 Column: Region
 Best: Latin America (1.405961881053661)
 Worst: Sinosphere (1.5543386994036725)
 Gap: 0.1483768183500116
 Remaining rows: 3473

Step 2
 Column: Age
 Best: 18-30 (1.439209464646139)
 Worst: 30-50 (1.5212325403033782)
 Gap: 0.0820230756572391
 Remaining rows: 900

Step 3
 Column: Gender
 Best: Man (1.4519318317311374)
 Worst: Woman (1.4992837797421115)
 Gap: 0.04735194801097409
 Remaining rows: 900

